# 실험 04: 실무형 고급 RAG 기법 (Grounding & Citation)

## 1. 실험 제목과 목표
- **제목**: 거짓말 못 하게 막고, 출처 밝히기
- **목표**: 교안 10번 슬라이드에 등장하는 고급 기법 중, 가장 빠르고 쉽게 적용할 수 있으면서도 실무 임팩트가 가장 큰 두 가지 기법(Grounding, Citation)을 프롬프트 엔지니어링으로 구현해 봅니다.

## 2. 실험 개요
1. **실험 1 (Grounding)**: 프롬프트에 강력한 족쇄를 채워서, 모델이 검색된 문서에 없는 내용은 절대 지어내지 말고 당당하게 "모른다"고 답하도록 강제합니다.
2. **실험 2 (Citation)**: 모델이 답변을 뱉을 때, 자기가 참고한 문서가 무엇인지(출처)를 사용자에게 명시적으로 보여주도록 만들어 신뢰도를 200% 끌어올립니다.

## 3. 라이브러리 import 및 이전 과정(Vector DB) 세팅
이전 노트북과 완벽히 동일하게 DB를 띄웁니다. 출처 표기를 위해 Document에 `metadata`를 추가합니다.

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0) # 환각 방지를 위해 창의성을 0으로 낮춤

# 꿀팁: 문서 객체에 metadata 딕셔너리를 넣으면 나중에 출처(Citation)로 써먹을 수 있습니다.
docs = [
    Document(
        page_content="[제2장 복지] 재택근무는 주 2회 가능하며, 팀장의 사전 승인이 필수입니다.", 
        metadata={"source": "사내 취업규칙 2024년판", "page": 12}
    ),
    Document(
        page_content="[제3장 휴가] 생일자에게는 오후 반차를 부여합니다.",
        metadata={"source": "임직원 복지 가이드라인", "page": 5}
    )
]
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 4. 실험 1: Grounding (근거 기반 족쇄 채우기)
03번 실험에서 모델이 엉뚱한 문서를 보고 헛소리를 하던 현상을 프롬프트 튜닝으로 잡아냅니다.

In [2]:
print("=== [1. Grounding이 적용된 깐깐한 프롬프트] ===")
grounding_prompt = ChatPromptTemplate.from_template("""
당신은 사내 규정을 안내하는 깐깐한 인사팀 직원입니다.
다음 제공된 [문서]만을 바탕으로 질문에 답하세요.

***매우 중요***:
1. 문서에 명시되지 않은 정보는 절대로 지어내지 마세요.
2. 질문에 대한 답을 문서에서 찾을 수 없다면, 반드시 "해당 내용은 현재 규정 문서에서 찾을 수 없습니다." 라고만 답변하세요.

[문서]
{context}

[질문]
{input}
""")

grounding_chain = create_retrieval_chain(
    retriever, 
    create_stuff_documents_chain(llm, grounding_prompt)
)

bad_question = "회식비 지원은 얼마나 돼?"
res = grounding_chain.invoke({"input": bad_question})

print("질문:", bad_question)
print("답변:", res["answer"])
print("\n-> 결과: 엉뚱한 문서를 가져오더라도, 프롬프트의 강력한 제약(Grounding) 덕분에 함부로 지어내지 않고 정직하게 모른다고 대답합니다. 실무에서 가장 중요한 패턴입니다.")

=== [1. Grounding이 적용된 깐깐한 프롬프트] ===
질문: 회식비 지원은 얼마나 돼?
답변: 해당 내용은 현재 규정 문서에서 찾을 수 없습니다.

-> 결과: 엉뚱한 문서를 가져오더라도, 프롬프트의 강력한 제약(Grounding) 덕분에 함부로 지어내지 않고 정직하게 모른다고 대답합니다. 실무에서 가장 중요한 패턴입니다.


## 5. 실험 2: Citation (출처 표기 방식)
프롬프트 안에서 `context`가 어떻게 조합되는지 살짝 커스텀하여, 모델이 출처를 보고 따라 말하게 만듭니다.

In [3]:
print("\n=== [2. 출처를 표기하는 Citation 프롬프트] ===")
citation_prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고하여 질문에 답하세요.
답변 끝에는 반드시 참고한 문서의 출처(Source)와 페이지(Page) 번호를 괄호 안에 적어주세요.

예시: 재택근무는 주 2회 가능합니다. (출처: 취업규칙, 12p)

[문서]
{context}

[질문]
{input}
""")

# 커스텀 문서 포맷팅: 기본적으로 create_stuff_documents_chain은 문서의 page_content만 텍스트로 합칩니다.
# metadata에 있는 source도 LLM이 볼 수 있게 합쳐주는 함수를 만들어 끼워넣어야 합니다.
def format_docs_with_source(docs):
    formatted_strings = []
    for doc in docs:
        # 본문 텍스트 뒤에 메타데이터를 텍스트로 붙여줌
        content = doc.page_content
        source = doc.metadata.get('source', '알 수 없음')
        page = doc.metadata.get('page', '알 수 없음')
        formatted_strings.append(f"내용: {content}\n메타데이터: [출처={source}, 페이지={page}]")
    return "\n---\n".join(formatted_strings)

# LangChain의 최신 LCEL 표기법을 사용하여 우리가 만든 커스텀 포맷 함수를 끼워넣습니다.
from langchain_core.runnables import RunnablePassthrough

# 검색기(Retriever)가 가져온 문서를 format_docs_with_source 함수를 거쳐서 context라는 변수로 둔갑시킵니다.
advanced_chain = (
    {"context": retriever | format_docs_with_source, "input": RunnablePassthrough()}
    | citation_prompt
    | llm
)

good_question = "나 다음 달 생일인데 휴가 있어?"
res = advanced_chain.invoke(good_question)

print("질문:", good_question)
print("\n[프롬프트에 실제로 들어간 context 확인]")
# retriever가 가져온 결과를 우리가 만든 포맷 함수에 넣어 눈으로 확인해봅니다.
print(format_docs_with_source(retriever.invoke(good_question)))

print("\n[최종 LLM 답변]")
print(res.content)
print("\n-> 결과: LLM이 텍스트(page_content)뿐만 아니라 우리가 뒤에 붙여준 메타데이터까지 읽고, 답변 끝에 멋지게 출처를 달아줍니다. 답변의 신뢰도가 수직 상승합니다.")


=== [2. 출처를 표기하는 Citation 프롬프트] ===
질문: 나 다음 달 생일인데 휴가 있어?

[프롬프트에 실제로 들어간 context 확인]
내용: [제3장 휴가] 생일자에게는 오후 반차를 부여합니다.
메타데이터: [출처=임직원 복지 가이드라인, 페이지=5]

[최종 LLM 답변]
생일자에게는 오후 반차가 부여됩니다. 따라서 다음 달 생일에 휴가를 받을 수 있습니다. (출처: 임직원 복지 가이드라인, 5p)

-> 결과: LLM이 텍스트(page_content)뿐만 아니라 우리가 뒤에 붙여준 메타데이터까지 읽고, 답변 끝에 멋지게 출처를 달아줍니다. 답변의 신뢰도가 수직 상승합니다.


## 6. 결과 해석

1. **Grounding의 중요성**: RAG 서비스가 실패하는 가장 큰 원인은 '모를 때 안 모른다고 하는 것'입니다. `temperature=0` 세팅과 깐깐한 프롬프트는 RAG 구축의 기본 중 기본입니다.
2. **Metadata 활용**: 문서 쪼개기(Chunking) 단계에서 문서의 제목, 작성자, 페이지 번호를 `metadata`에 꼼꼼히 넣어두면, 나중에 검색이나 출처 표기에 매우 강력하게 활용할 수 있습니다.
3. **LCEL의 강력함**: LangChain이 제공하는 붕어빵 틀(`create_retrieval_chain`)을 벗어나 `RunnablePassthrough` 등을 쓰면 중간 데이터를 입맛대로 조작할 수 있습니다.

## 7. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 3번 노트북에서 보이던 환각 증세가 강력한 프롬프트(Grounding)를 만나자 "모른다"고 당당하게 말하게 됨.
* `metadata`라는 개념을 사용하니 답변에 출처가 달려서, 사용자가 진짜인지 원본 문서를 찾아볼 수 있는 신뢰성을 확보함.

### 다음 개선 방향
* RAG 기초는 완벽히 마스터함.
* 더 나아가려면 교안 10번 슬라이드에 있는 `Hybrid Search(키워드+의미 검색)`, `Reranking(검색 순위 재정렬)` 등 검색기(Retriever) 자체의 성능을 올리는 기법들을 도입해야 함. (이는 심화 과정으로 남겨둠)